In [1]:
class settings:
  def __init__(self):
    self.iter=10
    self.results_dir="experiment_icd"
    self.dataset = "INA"
    self.metrics = ["AUC","AUPRC", "F1","PPV","Sensitivity", "Specificity", "NPV", "MCC", "Accuracy"]
    self.metrics_wilcoxon = ["AUC", "F1","Sensitivity", "Specificity", "MCC"]
    self.methods = ['STATIC','STATIC+BINARY','STATIC+FREQUENCY','STATIC+EVENT-CNT', 'STATIC+TF-IDF' ,'STATIC+BOW', 'STATIC+BiPadLSTM','STATIC+Med2Vec','STATIC+Dipole',
                    'STATIC+CEHR-BERT','STATIC+DOME','STATIC+GRU']

args = settings()

In [2]:
import pickle
# Caricamento dei risultati
results = {m: {} for m in args.methods}

for i in range(args.iter):
    for m in args.methods:
        meth = '_'.join(m.split('+'))
        path = (f"{args.results_dir}/results_{args.dataset}_{meth}_ITER{i}.pkl")
        with open(path, "rb") as f:
            results[m][i] = pickle.load(f)

In [3]:
import pandas as pd
dfs = {}
for metric in args.metrics:
    values = {method : [] for method in args.methods}
    for method in args.methods:
        lista = []
        for iter in range(args.iter):
            lista += results[method][iter][metric]
        values[method] = lista
    dfs[metric] = pd.DataFrame.from_dict(values)


In [4]:
import pandas as pd
from scripts.stats import friedman_test, wilcoxon_test, wilcoxon_ranking

alpha = 0.05
friedman_results = {}
wilcoxon_p_matrices = {}
wilcoxon_rankings = {}

for metric in args.metrics:
    df = dfs[metric]

    if "STATIC" in df.columns:
        df = df.drop(columns=["STATIC"])

    stat, p = friedman_test(df, alpha=alpha, debug=True)

    friedman_results[metric] = {
        "statistic": stat,
        "p_value": p,
        "global_diff": "YES" if p < alpha else "NO"
    }

    if p < alpha:
        p_matrix, reject, p_corr, results = wilcoxon_test(
            df,
            alpha=alpha,
            debug=True
        )

        ranking = wilcoxon_ranking(
            df,
            reject,
            higher_is_better=True
        )

        wilcoxon_p_matrices[metric] = p_matrix
        wilcoxon_rankings[metric] = ranking

friedman_df = pd.DataFrame(friedman_results).T


=== FRIEDMAN TEST ===
Statistic: 187.0404
p-value: 0.000000
➡️ SIGNIFICANT global differences

=== FRIEDMAN TEST ===
Statistic: 143.1452
p-value: 0.000000
➡️ SIGNIFICANT global differences

=== FRIEDMAN TEST ===
Statistic: 110.5838
p-value: 0.000000
➡️ SIGNIFICANT global differences

=== FRIEDMAN TEST ===
Statistic: 134.7127
p-value: 0.000000
➡️ SIGNIFICANT global differences

=== FRIEDMAN TEST ===
Statistic: 248.4346
p-value: 0.000000
➡️ SIGNIFICANT global differences

=== FRIEDMAN TEST ===
Statistic: 214.1934
p-value: 0.000000
➡️ SIGNIFICANT global differences

=== FRIEDMAN TEST ===
Statistic: 228.2941
p-value: 0.000000
➡️ SIGNIFICANT global differences

=== FRIEDMAN TEST ===
Statistic: 112.0849
p-value: 0.000000
➡️ SIGNIFICANT global differences

=== FRIEDMAN TEST ===
Statistic: 137.8194
p-value: 0.000000
➡️ SIGNIFICANT global differences


In [5]:
friedman_latex = friedman_df.copy()

friedman_latex["p_value"] = friedman_latex["p_value"].apply(
    lambda x: f"{x:.2e}"
)

friedman_latex["global_diff"] = friedman_latex["global_diff"].map({
    "YES": r"\checkmark",
    "NO": r"\ding{55}"
})

friedman_latex = friedman_latex.rename(columns={
    "statistic": "Statistic",
    "p_value": "$p$-value",
    "global_diff": "Difference"
})

friedman_latex = friedman_latex.T
latex_table = friedman_latex.T.to_latex(
    escape=False,
    caption="Friedman test results for all evaluation metrics.",
    label="tab:friedman",
    column_format="l" + "c"*len(friedman_latex.T.columns)
)

print(latex_table)

\begin{table}
\caption{Friedman test results for all evaluation metrics.}
\label{tab:friedman}
\begin{tabular}{lccc}
\toprule
 & Statistic & $p$-value & Difference \\
\midrule
AUC & 187.040408 & 8.07e-35 & \checkmark \\
AUPRC & 143.145249 & 9.55e-26 & \checkmark \\
F1 & 110.583782 & 4.07e-19 & \checkmark \\
PPV & 134.712707 & 5.09e-24 & \checkmark \\
Sensitivity & 248.434635 & 1.16e-47 & \checkmark \\
Specificity & 214.193400 & 1.75e-40 & \checkmark \\
NPV & 228.294150 & 1.96e-43 & \checkmark \\
MCC & 112.084928 & 2.03e-19 & \checkmark \\
Accuracy & 137.819396 & 1.18e-24 & \checkmark \\
\bottomrule
\end{tabular}
\end{table}



In [6]:
wilcoxon_rankings.keys()

dict_keys(['AUC', 'AUPRC', 'F1', 'PPV', 'Sensitivity', 'Specificity', 'NPV', 'MCC', 'Accuracy'])

In [7]:
ranking_table = pd.DataFrame()

for metric in args.metrics_wilcoxon:

    ranking = wilcoxon_rankings[metric].copy()

    ranking["rank"] = ranking["score"].rank(
        method="min",
        ascending=False
    ).astype(int)

    ranking_table[metric] = ranking["rank"]

ranking_table = ranking_table[args.metrics_wilcoxon]

# ordinamento gerarchico:
ranking_table = ranking_table.sort_values(
    by=args.metrics_wilcoxon,
    ascending=True      # rank 1 migliore
)
print(ranking_table.to_latex())

\begin{tabular}{lrrrrr}
\toprule
 & AUC & F1 & Sensitivity & Specificity & MCC \\
\midrule
STATIC+DOME & 1 & 1 & 1 & 8 & 1 \\
STATIC+FREQUENCY & 1 & 1 & 1 & 8 & 1 \\
STATIC+EVENT-CNT & 3 & 4 & 1 & 10 & 4 \\
STATIC+BOW & 4 & 3 & 5 & 2 & 3 \\
STATIC+TF-IDF & 5 & 8 & 6 & 7 & 6 \\
STATIC+BINARY & 5 & 11 & 4 & 11 & 11 \\
STATIC+Med2Vec & 7 & 4 & 8 & 2 & 6 \\
STATIC+BiPadLSTM & 8 & 4 & 7 & 2 & 5 \\
STATIC+Dipole & 9 & 9 & 8 & 2 & 6 \\
STATIC+CEHR-BERT & 10 & 4 & 8 & 1 & 9 \\
STATIC+GRU & 10 & 9 & 11 & 2 & 10 \\
\bottomrule
\end{tabular}

